# 04 · Baseline training

U-Net with a ResNet34 encoder pretrained on ImageNet (see model.py and the
baseline ADR). Each training component is validated in isolation here: a
forward-pass smoke test, class weights, the loss, and the per-class IoU metric.
The loop itself lives in cropweed_seg.engine and is run from scripts/train.py;
this notebook checks the pieces that loop depends on.

## Forward-pass smoke test

Confirm a real batch flows through the model and comes out the right shape
before any training. This checks plumbing, not quality: device, batch, model,
output. The predicted classes from an untrained model are meaningless, what
matters is that logits come out as (8, 3, 512, 512), one channel per class.

In [1]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from cropweed_seg.dataset import PeanutDataset
from cropweed_seg.transforms import build_transforms
from cropweed_seg.model import build_model

ROOT = Path.cwd().parent

# device auto-detection: cuda -> mps -> cpu
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device: {device}")

train_ds = PeanutDataset(ROOT, "train", build_transforms("train"))
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)

model = build_model().to(device)
model.eval()

img_batch, mask_batch = next(iter(train_loader))
img_batch = img_batch.to(device)

with torch.no_grad():
    logits = model(img_batch)

print(f"input:  {tuple(img_batch.shape)}")
print(f"logits: {tuple(logits.shape)}")
print(f"predicted classes (argmax): {torch.unique(logits.argmax(dim=1)).tolist()}")

device: mps
input:  (8, 3, 512, 512)
logits: (8, 3, 512, 512)
predicted classes (argmax): [0, 1, 2]


Logits are (8, 3, 512, 512), one channel per class at crop resolution, which is
the shape CrossEntropyLoss expects. Device is MPS. The plumbing works end to end.

## Class weights from the train split

The 84/12/3.7 imbalance means plain cross-entropy can score well by predicting
background everywhere and ignoring weed. Class weights counter this: rarer
classes cost more to get wrong. Weights are inverse frequency, normalized to
average 1, and computed from the train split only, never val or test, to avoid
leaking information (same care as ADR 0001).

In [2]:
import numpy as np
from PIL import Image

from cropweed_seg.masks import N_CLASSES, CLASS_NAMES

PROCESSED_LABELS = ROOT / "data" / "processed" / "labels"

# Class pixel counts from the TRAIN split only (no leakage from val/test)
train_stems = (ROOT / "data" / "splits" / "train.txt").read_text().split()

class_pixel_counts = np.zeros(N_CLASSES, dtype=np.int64)
for stem in train_stems:
    mask = np.asarray(Image.open(PROCESSED_LABELS / f"{stem}.png"))
    class_pixel_counts += np.bincount(mask.ravel(), minlength=N_CLASSES)

freq = class_pixel_counts / class_pixel_counts.sum()

# Inverse frequency, normalized so weights average to 1
inv_freq = 1.0 / freq
weights = inv_freq / inv_freq.mean()

print("Class balance and weights (train split):")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name:<11} freq {freq[i]:6.3%}   weight {weights[i]:6.2f}")

Class balance and weights (train split):
  background  freq 84.075%   weight   0.10
  crop        freq 12.194%   weight   0.68
  weed        freq 3.731%   weight   2.22


The weights follow the imbalance: weed at 2.22 against background at 0.10, so a
wrong weed pixel costs about 22 times a wrong background pixel. The train split
balance (84.1/12.2/3.7) matches the global figures from notebook 01, which
confirms the stratified split spread the classes evenly.

## Loss on a real batch

Two things to verify: the target cast to int64 (CrossEntropyLoss requires
integer class indices, not uint8), and that logits (N, C, H, W) and targets
(N, H, W) line up with no reshaping. The loss value itself is not meaningful
yet, it only needs to be a finite positive scalar.

In [3]:
import torch.nn as nn

class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Sanity check: loss runs on a real batch and returns a positive scalar
model.train()
img_batch = img_batch.to(device)
mask_batch = mask_batch.to(device).long()  # CrossEntropy wants int64 targets

logits = model(img_batch)
loss = criterion(logits, mask_batch)
print(f"loss on one batch: {loss.item():.4f}")

loss on one batch: 1.2698


Loss is a finite positive scalar. Shapes line up, weights apply, the cast works.
The training-side plumbing is ready.

## Per-class IoU metric

IoU is computed from a confusion matrix accumulated over the whole split, not
by averaging per-image IoU. With 29.5% of images lacking crop, per-image IoU
would be undefined for absent classes. The metric is validated on hand-checkable
cases: a perfect prediction, and one that misses all weed.

In [4]:
from cropweed_seg.metrics import ConfusionMatrixMetric

# Synthetic check: a perfect prediction must give IoU 1.0 on present classes
metric = ConfusionMatrixMetric()
target = torch.tensor([[[0, 1, 2], [0, 1, 2]]])  # (1, 2, 3)
metric.update(target.clone(), target.clone())     # predict == truth
print("perfect prediction:", metric.compute())

# A prediction that misses all weed (predicts background instead)
metric.reset()
pred = target.clone()
pred[pred == 2] = 0
metric.update(pred, target)
print("weed all wrong:    ", metric.compute())

perfect prediction: {'iou_background': 1.0, 'iou_crop': 1.0, 'iou_weed': 1.0, 'miou': 1.0}
weed all wrong:     {'iou_background': 0.5, 'iou_crop': 1.0, 'iou_weed': 0.0, 'miou': 0.5}


Both cases check out against hand calculation. The perfect prediction gives IoU
1.0 everywhere. When all weed is predicted as background, iou_weed drops to 0.0,
and iou_background drops to 0.5 because the misclassified weed pixels become
false positives for background. The metric captures that a weed error also
dirties the background score.

The lesson that justifies per-class reporting is visible here: mIoU of 0.5 hides
that weed is at 0.0. A model that ignores weed entirely still posts half a point
of mean IoU. This is why the three classes are always reported separately.

## Training run

The loop lives in cropweed_seg.engine (train_one_epoch, evaluate, class
weights, seeding, CSV logging) and is driven by scripts/train.py. The script
and this notebook import the same functions, so there is one source of truth
for the loop. Run the full training with:

```
uv run scripts/train.py
```

This seeds the run, trains for 15 epochs, keeps the best checkpoint by
validation mIoU, and writes per-epoch metrics to runs/baseline/metrics.csv.

## Baseline result

mIoU 0.768 at epoch 13, reproducible under seed 42:

- background 0.947
- crop 0.862
- weed 0.496

Background and crop plateau early. weed sits at roughly half their IoU and is
the bottleneck class the next experiments target. Validation loss flattens
around epochs 13 to 15 while train loss keeps falling, so the useful range is
the plateau near epoch 13, not more epochs. See ADR 0002 for the architecture
and checkpoint-selection decisions.